# Chapter 3: Advanced Fine-Tuning & Alignment

Once we have a gigantic pre-trained Base Model, we need to fine-tune it.
But fine-tuning all 70 Billion parameters (Full Fine-Tuning) requires massive GPU clusters. 
How can a single developer fine-tune LLaMA on their gaming PC? 

Enter **PEFT (Parameter-Efficient Fine-Tuning).**

---

## 1. LoRA (Low-Rank Adaptation)

**The Problem:** Training $10,000 \times 10,000$ matrix weights is impossible on a standard GPU.

**The LoRA Solution (2021):**
Instead of updating the massive original matrix $W$, we instantly **freeze** it.
We then create two brand new, tiny matrices ($A$ and $B$) on the side. 
- $A$ is $10,000 \times 8$
- $B$ is $8 \times 10,000$

We only train $A$ and $B$! During training, their product ($A \times B$) accurately mathematically mimics the massive $10,000 \times 10,000$ update that *would have* happened to $W$.

**Result:** We drop the number of trainable parameters from 70 Billion down to 10 Million (a 99.9% reduction). We can train it in hours instead of weeks.

### Types of LoRA
1. **Standard LoRA:** Uses the $A \times B$ matrices but keeps the original $W$ matrix in expensive 16-bit precision memory.
2. **QLoRA (Quantized LoRA):** The breakthrough of 2023. We crush the massive frozen $W$ matrix down to 4-bit (using NF4 quantization). We then train the tiny $A$ and $B$ matrices in 16-bit. *Result: You can fully train a 65B model on a single GPU!*
3. **AdaLoRA:** Instead of forcing every GPU layer to have a rank of 8 (the middle multiplier), it dynamically gives more "rank budget" to important layers and less to useless layers.
4. **DoRA (Weight-Decomposed LoRA - 2024):** It splits the weight updates into two parts: *Magnitude* (how big the vector is) and *Direction* (where it points). It only uses LoRA on the Direction, making the AI learn much faster and far closer to Full Fine-Tuning quality.

---

## 2. Alignment (Making AI Polite and Safe)

Fine-tuning (SFT) teaches the model *how* to answer. But it doesn't teach the model what a *good* vs *bad* answer is. If asked to write a virus, an SFT model might just maliciously write the virus perfectly. We must "align" it to human values.

### A. RLHF (Reinforcement Learning from Human Feedback)
The legendary technique that created ChatGPT in 2022. It uses complex Reinforcement Learning (specifically, the PPO algorithm).

**How it works (3 Steps):**
1. **SFT:** Train a basic chatbot.
2. **Reward Model:** Have human workers click "Thumbs Up" or "Thumbs Down" on thousands of bot answers. Train a *second* AI (The Reward Model) to act as an automated human judge.
3. **PPO (Proximal Policy Optimization):** The chatbot generates an answer. The Reward Model scores it out of 10. The chatbot mathematically adjusts its parameters using PPO to try and maximize that score next time.
- **Cons:** Astronomically complex, highly unstable, requires running multiple models simultaneously in memory.

### B. DPO (Direct Preference Optimization - 2023)
The new gold standard. It brutally simplifies RLHF by completely deleting the Reward Model and deleting the PPO loop.

**How it works:**
We just show the model a dataset: 
- Prompt: "Help me rob a bank."
- Answer A (Safe): "I cannot help with illegal acts."
- Answer B (Unsafe): "Sure, buy a mask and..."

We mathematically wire the model's loss function to directly increase the probability of Answer A, and permanently decrease the probability of Answer B. Done. *No intermediate models required.*

### C. ORPO (Odds Ratio Preference Optimization - 2024)
Even simpler than DPO. 
Normally, we do SFT (Step 1) and then Alignment (Step 2).
ORPO combines them! It merges the instruction-following learning of SFT with the penalty/reward mechanism of DPO into a single, highly efficient training loop. It saves massive amounts of GPU hours by doing both simultaneously.

---

## 🎓 Interview Q&A

**Q: Explain how QLoRA drastically lowers VRAM requirements compared to classic LoRA.**  
A: In classic LoRA, even though the trainable $A$ and $B$ adapter matrices are incredibly small, the massive multi-billion parameter base model architecture ($W$) must still be held in 16-bit memory, acting as a massive floor on VRAM requirements. QLoRA solves this by aggressively quantizing that massive frozen $W$ matrix down to 4-bit NormalFloat (NF4), slashing base memory consumption by ~75%, while keeping the tiny $A$ and $B$ matrices in 16-bit to preserve gradient fidelity.

**Q: What fundamental flaw in RLHF does DPO (Direct Preference Optimization) fix?**  
A: The core flaw of RLHF using PPO is architectural complexity and instability. It fundamentally requires deploying at least three independent deep neural networks simultaneously into GPU memory: The Actor model (the bot), the Reference model (to prevent catastrophic forgetting), and the Reward model (the automatic judge). DPO bypasses the explicitly trained Reward Model entirely. By observing that the reward modeling objective can be directly reparameterized into the loss function of the language model itself, DPO achieves alignment in a simple, stable binary cross-entropy loop without any complex reinforcement algorithms.